## Homework: Vector Search

* convert text into vectors
* perform similarity search
* hybrid search = keyword search + vector search

## Setup

In this homework we use the lightweight ONNX `Embedder` instead of sentence-transformers.
It needs no PyTorch and no CUDA — about 30x smaller installation.

Install dependencies (run in terminal):

```bash
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
```

Download helper scripts (run in terminal):

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py
```

Download the ONNX model (run in terminal):

```bash
uv run python download.py
```

## Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (`v[0]`)?

* [] -0.31
* [x] -0.02
* [] 0.12
* [] 0.44

In [1]:
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"
v1 = embed.encode(query)
v1[0]

2026-07-03 21:41:43.037010982 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


np.float64(-0.02058203437252893)

## Loading the data

* We pull the lesson pages from the course repository pinned to commit `8c1834d` so everyone works with the same data.
* Each document is a dictionary with a `filename` and `content`, and there are 72 pages.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(len(documents))

72


In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?

* [] 0.07
* [x] 0.37
* [] 0.68
* [] 0.92

In [4]:
filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

for doc in documents:
    if doc["filename"] == filename:
        content = doc["content"]

v2 = embed.encode(content)
v1.dot(v2)

np.float64(0.36107026789538205)

## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

Chunk the pages:

```python
chunks = chunk_documents(documents, size=2000, step=1000)
```

Embed every chunk, stack into matrix `X`, and score against the Q1 query:

```python
scores = X.dot(v)
```

Which file does the highest-scoring chunk belong to?

* [] `02-vector-search/lessons/03-embeddings-dataset.md`
* [] `02-vector-search/lessons/06-rag-vector.md`
* [x] `02-vector-search/lessons/07-sqlitesearch-vector.md`
* [] `02-vector-search/lessons/09-onnx-embedder.md`

In [5]:
from gitsource import chunk_documents
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

vectors = []

for chunk in chunks:
    chunk_vector = embed.encode(chunk["content"])
    vectors.append(chunk_vector)


X = np.array(vectors)
print(X.shape)  # shape = (num_chunks, 384)


(295, 384)


In [6]:
scores = X.dot(v1)
index_best_match = scores.argmax()

chunks[index_best_match]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Q4. Vector search with minsearch

Use `VectorSearch` from minsearch and run a search for:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

* [] `02-vector-search/lessons/04-vector-search.md`
* [x] `04-evaluation/lessons/05-search-metrics.md`
* [] `04-evaluation/lessons/13-llm-as-judge.md`
* [] `05-monitoring/lessons/04-metrics.md`

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

In [ ]:
query4 = "What metric do we use to evaluate a search engine?"
query4_vector = embed.encode(query4)

results = vindex.search(query4_vector, num_results=1)
results

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

## Q5. Text search vs vector search

Index the same chunks with `Index` from minsearch (use `content` as a text field).

Run both searches for:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each. Which file shows up in the vector results but **not** in the text results?

* [] `02-vector-search/lessons/01-intro.md`
* [] `02-vector-search/lessons/02-embeddings.md`
* [x] `02-vector-search/lessons/08-pgvector.md`
* [] `03-orchestration/lessons/05-rag.md`

In [ ]:
query5 = "How do I store vectors in PostgreSQL?"
query5_vector = embed.encode(query5)

vector_results = vindex.search(query5_vector, num_results=5)
vector_results_list = [v["filename"] for v in vector_results]
vector_results_list


['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [ ]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["course"])

text_index.fit(chunks)
text_search_results = text_index.search(query5, num_results=5)

text_results_list = [t["filename"] for t in text_search_results]
text_results_list

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [11]:
# in vector but not in text search
print(set(vector_results_list) - set(text_results_list))


{'02-vector-search/lessons/08-pgvector.md'}


## Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. We can combine them — this is called **hybrid search**.

We use **Reciprocal Rank Fusion (RRF)** to merge ranked lists. It ignores raw scores and looks only at rank position:

```
RRF(d) = sum over lists of  1 / (k + rank(d))
```

The constant `k = 60` comes from the original RRF paper. A document found by both searches collects a score from each list, so it ranks higher than one found by only one search.

Run `"How do I give the model access to tools?"` with both searches and fuse with `rrf`.

Which file is ranked first after RRF?

* [] `01-agentic-rag/lessons/01-intro.md`
* [x] `01-agentic-rag/lessons/13-function-calling.md`
* [] `01-agentic-rag/lessons/14-agentic-loop.md`
* [] `01-agentic-rag/lessons/16-other-frameworks.md`

Notice that this file is not first in either search on its own — it wins because it ranks high in both.

In [12]:
def rrf(ranked_lists: list[list], k: int = 60) -> list[dict]:
    """Implement RRF the RRF formula: 1 / (k + rank)"""
    scores = {}
    docs = {}
    for r_list in ranked_lists:
        for rank, item in enumerate(r_list, start=1):
            filename = item["filename"]
            scores[filename] = scores.get(filename, 0.0) + (1.0 / (k + rank))
            if filename not in docs:
                docs[filename] = item

    # Return sorted by score descending
    return sorted(docs.values(), key=lambda x: scores[x["filename"]], reverse=True)

In [14]:
query6 = "How do I give the model access to tools?"
query6_vector = embed.encode(query6)

v_results = vindex.search(query6_vector, num_results=5)
t_results = text_index.search(query6, num_results=5)

fused = rrf([v_results, t_results])
fused

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 

## Submit the results

* Submit your results here: https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw2
* It's possible your answers won't match exactly. If so, select the closest one.